# CrashCase: train the damage segmentation model

Runs the repo's own `training/prepare_cardd.py` and `training/train.py` on a Kaggle GPU with the CarDD dataset attached (`issamjebnouni/cardd`). Needs a GPU and internet turned on.

Outputs in `/kaggle/working`: `damage-seg.pt` (copy to `models/`), `metrics.json`, training plots, and the MLflow run (`mlflow.db` + `mlruns.zip`).

In [ ]:
# Branch of the repo to train from
REF = "fix/local-setup"

!nvidia-smi -L
!git clone --depth 1 --branch {REF} https://github.com/uzairahm290/CrashCase-AI /tmp/CrashCase-AI
!pip install -q "ultralytics>=8.2" "mlflow>=2.12"

Lay the Kaggle copy out as `datasets/CarDD_COCO`, which is what the converter expects, then convert to YOLO-seg.

In [ ]:
import glob
import os
from pathlib import Path

os.chdir("/tmp/CrashCase-AI")
src = Path(glob.glob("/kaggle/input/**/train.json", recursive=True)[0]).parent
coco = Path("datasets/CarDD_COCO")
(coco / "annotations").mkdir(parents=True, exist_ok=True)
for split in ("train", "val", "test"):
    (coco / f"{split}2017").symlink_to(src / split)
    (coco / "annotations" / f"instances_{split}2017.json").symlink_to(src / f"{split}.json")

import subprocess

# subprocess with check=True stops the notebook on failure; a plain !python would carry on
subprocess.run(["python", "training/prepare_cardd.py"], check=True)

Train with the settings in `params.yaml`. MLflow logs to a local SQLite store so the run can be imported later.

In [ ]:
os.environ.update(MLFLOW_TRACKING_URI="sqlite:////tmp/mlflow.db", WANDB_MODE="disabled")
subprocess.run(["python", "training/train.py"], check=True)

Keep only what is needed: the weights, metrics, plots and the MLflow run.

In [ ]:
import json
import shutil

out = Path("/kaggle/working")
shutil.copy2("models/damage-seg.pt", out)
shutil.copy2("metrics.json", out)
(out / "plots").mkdir(exist_ok=True)
for plot in glob.glob("runs/**/damage-seg/*.png", recursive=True):
    shutil.copy2(plot, out / "plots")
shutil.copy2("/tmp/mlflow.db", out)
shutil.make_archive(str(out / "mlruns"), "zip", "mlruns")
print(json.dumps(json.loads(Path("metrics.json").read_text()), indent=2))